# Phase 4 — Supervised MLP Training
**RealCentric Generator-Agnostic Deepfake Detection**  
SVKM AI/ML HPC Cluster

Trains a binary MLP classifier on the 718-dim fused features.

**Input:** `/data/mpstme-naman/deepfake_detection/data/features/Z_train.npy`  
**Output:** `/data/mpstme-naman/deepfake_detection/checkpoints/mlp_supervised_best.pt`  
**Estimated time:** 10–20 minutes

## Step 1 — Load Features

In [ ]:
import sys, numpy as np, torch
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path

BASE     = Path('/data/mpstme-naman/deepfake_detection')
FEAT_DIR = BASE / 'data' / 'features'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'
for d in [CKPT_DIR, RES_DIR]: d.mkdir(parents=True, exist_ok=True)

Z_train = np.load(FEAT_DIR/'Z_train.npy');  y_train = np.load(FEAT_DIR/'y_train.npy')
Z_val   = np.load(FEAT_DIR/'Z_val.npy');    y_val   = np.load(FEAT_DIR/'y_val.npy')

print(f'  Train : {Z_train.shape}   real={(y_train==0).sum():,}  fake={(y_train==1).sum():,}')
print(f'  Val   : {Z_val.shape}   real={(y_val==0).sum():,}  fake={(y_val==1).sum():,}')
print(f'  GPU   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Step 2 — Initialise Trainer

In [ ]:
from config.config_loader import load_config
from src.models.mlp_classifier import MLPTrainer

cfg     = load_config()
trainer = MLPTrainer(cfg=cfg, input_dim=Z_train.shape[1])
print(f'  Input  : {Z_train.shape[1]} dims')
print(f'  Hidden : 256 → 64 (ReLU + Dropout 0.3)')
print(f'  Output : 1 (Sigmoid → probability of fake)')
print(f'  Loss   : Binary Cross-Entropy')
print(f'  Optim  : Adam  lr=1e-4  weight_decay=1e-4')
print(f'  Epochs : 50  (early stop patience=10)')

## Step 3 — Train

In [ ]:
best_auc = trainer.train(
    Z_train, y_train, Z_val, y_val,
    checkpoint_dir=str(CKPT_DIR),
    run_name='mlp_supervised'
)
print(f'\n  ✓  Best validation AUC: {best_auc:.4f}')
print(f'  ✓  Saved → {CKPT_DIR}/mlp_supervised_best.pt')

## Step 4 — Evaluate on CelebDF Test Set

In [ ]:
Z_test = np.load(FEAT_DIR/'Z_test.npy'); y_test = np.load(FEAT_DIR/'y_test.npy')
m = trainer.evaluate(Z_test, y_test)
print('=' * 50)
print('  CelebDF Test Set Results')
print('=' * 50)
for k in ['accuracy','auc','precision','recall','f1']:
    print(f'  {k:<12}: {m[k]:.4f}')
print(f'\n  TP={m["tp"]}  FP={m["fp"]}  FN={m["fn"]}  TN={m["tn"]}')

## Step 5 — Cross-Dataset Generalisation

In [ ]:
print('Cross-dataset test:')
print('=' * 55)
for name, zf, yf in [('FaceForensics++','Z_ff.npy','y_ff.npy'),('Stable Diffusion','Z_sd.npy','y_sd.npy')]:
    Z = np.load(FEAT_DIR/zf); y = np.load(FEAT_DIR/yf)
    if len(np.unique(y)) < 2:
        prob = trainer.predict_proba(Z)
        print(f'  {name:<20} mean_score={prob.mean():.4f}  (fake-only dataset)')
    else:
        m = trainer.evaluate(Z, y)
        print(f'  {name:<20} ACC={m["accuracy"]:.2f}%  AUC={m["auc"]:.4f}  F1={m["f1"]:.4f}')

## Step 6 — Training Curve Plot

In [ ]:
import matplotlib.pyplot as plt
if hasattr(trainer, '_history') and trainer._history:
    h = trainer._history
    epochs = range(1, len(h['train_loss'])+1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, h['train_loss'], color='steelblue', label='Train Loss')
    ax1.plot(epochs, h['val_loss'],   color='crimson',   label='Val Loss')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, h['val_auc'], color='green', label='Val AUC')
    ax2.set_title('Validation AUC'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    out = RES_DIR/'mlp_training_curve.png'
    plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
    print(f'  ✓  {out}')

## ✅ Phase 4 Complete

**Saved:** `checkpoints/mlp_supervised_best.pt`

**Next:** Open `04_train_unsupervised.ipynb`